In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/aymanalzahrani7/graduation-projectd/Graduation Project/Cleaned Data From Youtube/( المنطقة الشمالية ) Youtube Data -  After Cleaning/تبوك/تبوك1_textready.xlsx
/kaggle/input/datasets/aymanalzahrani7/graduation-projectd/Graduation Project/Cleaned Data From Youtube/( المنطقة الشمالية ) Youtube Data -  After Cleaning/تبوك/منطقة بجدة البرية_textready.xlsx
/kaggle/input/datasets/aymanalzahrani7/graduation-projectd/Graduation Project/Cleaned Data From Youtube/( المنطقة الشمالية ) Youtube Data -  After Cleaning/تبوك/وادي الديسة_textready.xlsx
/kaggle/input/datasets/aymanalzahrani7/graduation-projectd/Graduation Project/Cleaned Data From Youtube/( المنطقة الشمالية ) Youtube Data -  After Cleaning/تبوك/جبل اللوز_textready.xlsx
/kaggle/input/datasets/aymanalzahrani7/graduation-projectd/Graduation Project/Cleaned Data From Youtube/( المنطقة الشمالية ) Youtube Data -  After Cleaning/تبوك/تبوك_textready.xlsx
/kaggle/input/datasets/aymanalzahrani7/graduation-projectd/Graduation

In [21]:
import os, glob, re, pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix,f1_score
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [22]:
import os, glob, re
import numpy as np
import pandas as pd

from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks

# لو تعمل داخل Jupyter
from IPython.display import display

In [61]:
# =========================================================
# (0) PATH + SETTINGS  (ALL REGIONS) + QUICK STRUCTURE CHECK
# =========================================================
ALL_REGIONS_ROOT = "/kaggle/input/datasets/aymanalzahrani7/graduation-projectd/Graduation Project/Cleaned Data From Google Maps/"
TEXT_COL = "Text_TR"  # ✅ العمود المراد العمل عليه
STAR_CANDIDATES = {"stars", "Stars"}
print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir:", os.path.isdir(ALL_REGIONS_ROOT))

region_dirs = sorted([d for d in glob.glob(os.path.join(ALL_REGIONS_ROOT, "*")) if os.path.isdir(d)])
print("Region folders found:", len(region_dirs))
print("Regions:", [os.path.basename(d) for d in region_dirs])

# عرض سريع لأول 3 مناطق: عدد المدن داخل كل منطقة
for rd in region_dirs[:3]:
    city_dirs = sorted([d for d in glob.glob(os.path.join(rd, "*")) if os.path.isdir(d)])
    print(f"  - {os.path.basename(rd)}: cities={len(city_dirs)} (sample: {[os.path.basename(x) for x in city_dirs[:5]]})")



[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir: True
Region folders found: 5
Regions: ['( المنطقة الجنوبية ) Google Maps Data -  After Cleaning', '( المنطقة الشرقية ) Google Maps Data -  After Cleaning', '( المنطقة الشمالية ) Google Maps Data -  After Cleaning', '( المنطقة الغربية ) Google Maps Data -  After Cleaning', '( المنطقة الوسطى ) Google Maps Data -  After Cleaning']
  - ( المنطقة الجنوبية ) Google Maps Data -  After Cleaning: cities=4 (sample: ['منطقة الباحة', 'منطقة جازان', 'منطقة عسير', 'منطقة نجران'])
  - ( المنطقة الشرقية ) Google Maps Data -  After Cleaning: cities=6 (sample: ['الاحساء', 'الجبيل', 'الخبر', 'الدمام', 'الظهران'])
  - ( المنطقة الشمالية ) Google Maps Data -  After Cleaning: cities=5 (sample: ['تبوك', 'محافظة العلا', 'منطقة الجوف', 'منطقة الحدود الشمالية - عرعر', 'منطقة حائل'])


In [62]:
# =========================================================
# Helpers: extract region name (inside parentheses) + city
# =========================================================
PAREN_RE = re.compile(r"\((.*?)\)")

def extract_region_from_folder(region_folder_name: str) -> str:
    """
    يستخرج النص بين ( ) من اسم مجلد المنطقة.
    مثال: "( المنطقة الغربية ) Google Maps Data - After Cleaning" -> "المنطقة الغربية"
    إذا لم توجد أقواس يرجع اسم المجلد نفسه.
    """
    m = PAREN_RE.search(region_folder_name)
    if m:
        return m.group(1).strip()
    return region_folder_name.strip()

def list_region_folders(root_folder: str):
    region_dirs = sorted([d for d in glob.glob(os.path.join(root_folder, "*")) if os.path.isdir(d)])
    return region_dirs

def list_city_folders(region_dir: str):
    return sorted([d for d in glob.glob(os.path.join(region_dir, "*")) if os.path.isdir(d)])

def list_files_in_city(city_dir: str):
    return (
        glob.glob(os.path.join(city_dir, "*.xlsx")) +
        glob.glob(os.path.join(city_dir, "*.xls")) +
        glob.glob(os.path.join(city_dir, "*.csv"))
    )

def read_any_file(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    elif ext == ".csv":
        # ترميزات عربية شائعة
        try:
            df = pd.read_csv(path, encoding="utf-8")
        except UnicodeDecodeError:
            try:
                df = pd.read_csv(path, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(path, encoding="cp1256")
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    df.columns = [str(c).strip() for c in df.columns]
    return df

def standardize_star_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    - يقبل Stars / stars / ... ويوحّدها إلى عمود اسمه 'Stars' إن وجد.
    - لا يغير القيم الآن.
    """
    colmap = {c.lower(): c for c in df.columns}
    # إذا موجود Stars بالاسم الصحيح خلاص
    if "Stars" in df.columns:
        return df

    # ابحث عن أي عمود اسمه stars case-insensitive
    if "stars" in colmap:
        df = df.rename(columns={colmap["stars"]: "Stars"})
        return df

    # مرونة إضافية: إذا فيه rating مثلا
    for cand in list(STAR_CANDIDATES):
        key = cand.lower()
        if key in colmap:
            df = df.rename(columns={colmap[key]: "Stars"})
            return df

    return df  # لم نجد عمود نجوم

In [63]:

# =========================================================
# [STEP 0] Path checks + show regions/cities counts
# =========================================================
print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir :", os.path.isdir(ALL_REGIONS_ROOT))

region_dirs = list_region_folders(ALL_REGIONS_ROOT)
print("Region folders found:", len(region_dirs))

# عرض أسماء المناطق بصيغة الاسم داخل الأقواس
for rd in region_dirs:
    folder_name = os.path.basename(rd)
    region_name = extract_region_from_folder(folder_name)
    print(f"- Folder: {folder_name}  --> Region: {region_name}")



[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir : True
Region folders found: 5
- Folder: ( المنطقة الجنوبية ) Google Maps Data -  After Cleaning  --> Region: المنطقة الجنوبية
- Folder: ( المنطقة الشرقية ) Google Maps Data -  After Cleaning  --> Region: المنطقة الشرقية
- Folder: ( المنطقة الشمالية ) Google Maps Data -  After Cleaning  --> Region: المنطقة الشمالية
- Folder: ( المنطقة الغربية ) Google Maps Data -  After Cleaning  --> Region: المنطقة الغربية
- Folder: ( المنطقة الوسطى ) Google Maps Data -  After Cleaning  --> Region: المنطقة الوسطى


In [64]:
# =========================================================
# (1) SCAN + READ PER REGION (chunked) + AVAILABILITY REPORT
#    - لا يعلّق: يقرأ منطقة-منطقة، وملف-ملف
#    - يعطيك تقرير توفر الأعمدة لكل منطقة
# =========================================================
def scan_availability(root_folder: str):
    """
    يعرض تقرير توفر الأعمدة (Text_TR / Stars) لكل منطقة قبل دمج كل البيانات.
    """
    rows = []
    for rd in list_region_folders(root_folder):
        region_folder = os.path.basename(rd)
        region_name = extract_region_from_folder(region_folder)

        city_dirs = list_city_folders(rd)
        n_cities = len(city_dirs)
        files = []
        for cd in city_dirs:
            files.extend(list_files_in_city(cd))

        rows.append({
            "Region_Folder": region_folder,
            "Region_Name": region_name,
            "Cities": n_cities,
            "Files": len(files),
            "Sample_File": files[0] if files else None
        })
    return pd.DataFrame(rows).sort_values("Files", ascending=False).reset_index(drop=True)

df_av = scan_availability(ALL_REGIONS_ROOT)
print("\n[STEP 1A] Availability (folders/files) per region:")
display(df_av)

def read_all_regions_chunked(root_folder: str, max_files_per_region=None, keep_columns=None):
    """
    - يقرأ كل المناطق على شكل chunks (منطقة-منطقة).
    - max_files_per_region: لتجربة/تقليل الحمل (None = الكل).
    - keep_columns: لو تبغى تحتفظ بأعمدة محددة فقط لتخفيف الذاكرة.
    - يرجع:
        df_all (اختياري) + df_region_report (تقرير توفر الأعمدة لكل منطقة)
    """
    all_frames = []
    report_rows = []

    region_dirs = list_region_folders(root_folder)

    for r_idx, rd in enumerate(region_dirs, 1):
        region_folder = os.path.basename(rd)
        region_name = extract_region_from_folder(region_folder)
        city_dirs = list_city_folders(rd)

        region_files = []
        for cd in city_dirs:
            region_files.extend(list_files_in_city(cd))

        if max_files_per_region is not None:
            region_files = region_files[:max_files_per_region]

        print(f"\n[STEP 1] Region {r_idx}/{len(region_dirs)}: {region_name} | cities={len(city_dirs)} | files={len(region_files)}")

        n_rows_region = 0
        has_text = 0
        has_stars = 0
        read_ok = 0
        read_err = 0

        for i, p in enumerate(region_files, 1):
            try:
                df = read_any_file(p)
                df = standardize_star_column(df)

                city_name = os.path.basename(os.path.dirname(p))  # مجلد المدينة
                df["Region_Folder"] = region_folder
                df["Region_Name"] = region_name
                df["City_Folder"] = city_name
                df["Source_File"] = os.path.basename(p)
                df["__path__"] = p

                # توفر الأعمدة
                if TEXT_COL in df.columns:
                    has_text += 1
                if "Stars" in df.columns:
                    has_stars += 1

                # لتخفيف الذاكرة: احتفظ بأعمدة معينة فقط
                if keep_columns is not None:
                    keep = [c for c in keep_columns if c in df.columns]
                    # دائما احتفظ بالميتا
                    meta = ["Region_Folder","Region_Name","City_Folder","Source_File","__path__"]
                    keep = list(dict.fromkeys(keep + meta))  # unique preserve order
                    df = df[keep].copy()

                n_rows_region += len(df)
                all_frames.append(df)
                read_ok += 1

                if i <= 2:
                    print(f"  sample file [{i}] {region_name}/{city_name}/{os.path.basename(p)} rows={len(df)}")
                if i % 50 == 0:
                    print(f"  progress: {i}/{len(region_files)} files")

            except Exception as e:
                read_err += 1

        report_rows.append({
            "Region_Name": region_name,
            "Region_Folder": region_folder,
            "Cities": len(city_dirs),
            "Files_Read_OK": read_ok,
            "Files_Read_Err": read_err,
            "Files_with_Text_TR": has_text,
            "Files_with_Stars": has_stars,
            "Total_Rows_This_Region": n_rows_region
        })

    df_region_report = pd.DataFrame(report_rows).sort_values("Total_Rows_This_Region", ascending=False).reset_index(drop=True)

    df_all = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
    return df_all, df_region_report


# ===== تشغيل القراءة (خفيف) =====
# keep_columns لتقليل الذاكرة: خذ فقط ما تحتاجه
KEEP_COLS = [TEXT_COL, "Stars"]  # بعد التوحيد سيصبح Stars موجود إذا كان موجوداً
df1, df_region_report = read_all_regions_chunked(
    ALL_REGIONS_ROOT,
    max_files_per_region=None,     # ضع رقم مثل 50 للتجربة أو لتقليل الحمل
    keep_columns=KEEP_COLS
)

print("\n[STEP 1B] Column availability per region (after scan/read):")
display(df_region_report)

print("\n[STEP 1 RESULT] df1 shape:", df1.shape)
print("Columns:", df1.columns.tolist())


[STEP 1A] Availability (folders/files) per region:


,Region_Folder,Region_Name,Cities,Files,Sample_File
0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,4,141,/kaggle/input/datasets/aymanalzahrani7/graduat...
1,( المنطقة الوسطى ) Google Maps Data - After C...,المنطقة الوسطى,2,85,/kaggle/input/datasets/aymanalzahrani7/graduat...
2,( المنطقة الغربية ) Google Maps Data - After ...,المنطقة الغربية,6,85,/kaggle/input/datasets/aymanalzahrani7/graduat...
3,( المنطقة الشرقية ) Google Maps Data - After ...,المنطقة الشرقية,6,74,/kaggle/input/datasets/aymanalzahrani7/graduat...
4,( المنطقة الشمالية ) Google Maps Data - After...,المنطقة الشمالية,5,57,/kaggle/input/datasets/aymanalzahrani7/graduat...



[STEP 1] Region 1/5: المنطقة الجنوبية | cities=4 | files=141
  sample file [1] المنطقة الجنوبية/منطقة الباحة/حديقة الفراشة بالمندق_textready.xlsx rows=2715
  sample file [2] المنطقة الجنوبية/منطقة الباحة/أكواخ سار الريفية_textready.xlsx rows=440
  progress: 50/141 files
  progress: 100/141 files

[STEP 1] Region 2/5: المنطقة الشرقية | cities=6 | files=74
  sample file [1] المنطقة الشرقية/الاحساء/قصر محيرس الاثري_textready.xlsx rows=4489
  sample file [2] المنطقة الشرقية/الاحساء/مقهى السيد_textready.xlsx rows=1376
  progress: 50/74 files

[STEP 1] Region 3/5: المنطقة الشمالية | cities=5 | files=57
  sample file [1] المنطقة الشمالية/تبوك/منتجع فيروز_final_final_textready.xlsx rows=425
  sample file [2] المنطقة الشمالية/تبوك/تبوك بارك_final_final_textready.xlsx rows=7242
  progress: 50/57 files

[STEP 1] Region 4/5: المنطقة الغربية | cities=6 | files=85
  sample file [1] المنطقة الغربية/الشعيبة/بحر الشعيبة منطقة السباحة _final_textready.xlsx rows=622
  sample file [2] المنطقة الغربية/الش

,Region_Name,Region_Folder,Cities,Files_Read_OK,Files_Read_Err,Files_with_Text_TR,Files_with_Stars,Total_Rows_This_Region
0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,4,141,0,141,140,276473
1,المنطقة الشرقية,( المنطقة الشرقية ) Google Maps Data - After ...,6,74,0,74,74,255183
2,المنطقة الوسطى,( المنطقة الوسطى ) Google Maps Data - After C...,2,85,0,85,85,226035
3,المنطقة الغربية,( المنطقة الغربية ) Google Maps Data - After ...,6,85,0,85,85,186830
4,المنطقة الشمالية,( المنطقة الشمالية ) Google Maps Data - After...,5,57,0,57,57,72074



[STEP 1 RESULT] df1 shape: (1016595, 7)
Columns: ['Text_TR', 'Stars', 'Region_Folder', 'Region_Name', 'City_Folder', 'Source_File', '__path__']


In [68]:
 df1['Stars'].value_counts()

Stars
5.0    646067
4.0    166508
3.0    103035
1.0     65871
2.0     34734
Name: count, dtype: int64

In [69]:
df1.head(10)

,Text_TR,Stars,Region_Folder,Region_Name,City_Folder,Source_File,__path__
0,جميلة جدا,5.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
1,تحتاج صيانه,3.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
2,مكان جميل وشعبي,5.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
4,NaN,4.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
5,NaN,4.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
6,NaN,5.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
7,NaN,4.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
8,NaN,5.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
9,NaN,5.0,( المنطقة الجنوبية ) Google Maps Data - After...,المنطقة الجنوبية,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...


In [70]:
# =========================================================
# (2) FILTER ONLY TRULY-EMPTY Text_TR (NaN + "" + "nan" + "[]" ...)
# =========================================================
print(f"\n[STEP 2] Using {TEXT_COL} + strong empty filter")

# ✅ الداتا الجديدة
if TEXT_COL not in df1.columns:
    raise ValueError(f"عمود {TEXT_COL} غير موجود في البيانات.")

before2 = len(df1)

# لا نحول إلى string الآن (مهم)
s = df1[TEXT_COL]

empty_like = {
    "", "nan", "NaN", "none", "None", "NONE",
    "<NA>", "[]", "[ ]", "{}", "null", "NULL"
}

empty_mask = (
    s.isna() |
    s.astype(str).str.strip().isin(empty_like)
)

empty_count = int(empty_mask.sum())

# ✅ الناتج الجديد
df2 = df1.loc[~empty_mask].copy()

# العمود الذي سيستخدم لاحقاً للمودل
df2["TEXT_FOR_MODEL"] = df2[TEXT_COL].astype(str).str.strip()

print("\n[STEP 2 RESULT]")
print("Rows before:", before2)
print("Empty-like rows:", empty_count)
print("Rows after:", len(df2))

print("\nSample empty-like values:")
print(s.loc[empty_mask].head(10).astype(str).tolist())


[STEP 2] Using Text_TR + strong empty filter

[STEP 2 RESULT]
Rows before: 1016595
Empty-like rows: 505054
Rows after: 511541

Sample empty-like values:
['nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan']


In [72]:
# =========================================================
# (3) BINARY LABELS FROM STARS
#   - سلبي: 1-2
#   - إيجابي: 4-5
#   - نستبعد: 3 (محايد)
# =========================================================
print("\n[STEP 3] Binary labels from Stars (drop 3-stars)")

# تأكد من وجود العمود
if "Stars" not in df2.columns:
    raise ValueError("عمود Stars غير موجود بعد STEP 2.")

# تحويل النجوم إلى رقم (يتحمل نصوص)
df2["Stars_num"] = pd.to_numeric(df2["Stars"], errors="coerce")

before3 = len(df2)

# حذف القيم غير الصالحة
df3 = df2.dropna(subset=["Stars_num"]).copy()

# الاحتفاظ فقط 1,2,4,5
df3 = df3[df3["Stars_num"].isin([1, 2, 4, 5])].copy()

# ترميز ثنائي
df3["y_bin"] = (df3["Stars_num"] >= 4).astype(int)

print("\n[STEP 3 RESULT]")
print("Rows before:", before3)
print("Rows after binary filter:", len(df3))

print("\nBinary distribution (overall):")
print(df3["y_bin"].value_counts().rename({0:"سلبي", 1:"ايجابي"}))


# =========================================================
# ⭐ إحصائيات لكل منطقة (جديد)
# =========================================================
if "Region_Name" in df3.columns:

    print("\nBinary distribution per region:")

    region_stats = (
        df3.groupby(["Region_Name", "y_bin"])
           .size()
           .unstack(fill_value=0)
           .rename(columns={0:"سلبي", 1:"ايجابي"})
           .sort_values("ايجابي", ascending=False)
    )

    display(region_stats)

    print("\nTotal rows per region:")
    display(df3["Region_Name"].value_counts().to_frame("rows"))


[STEP 3] Binary labels from Stars (drop 3-stars)

[STEP 3 RESULT]
Rows before: 511541
Rows after binary filter: 456176

Binary distribution (overall):
y_bin
ايجابي    395241
سلبي       60935
Name: count, dtype: int64

Binary distribution per region:


y_bin,سلبي,ايجابي
Region_Name,,
المنطقة الجنوبية,19835,114740
المنطقة الشرقية,13199,100391
المنطقة الوسطى,13546,90870
المنطقة الغربية,9514,59111
المنطقة الشمالية,4841,30129



Total rows per region:


,rows
Region_Name,
المنطقة الجنوبية,134575
المنطقة الشرقية,113590
المنطقة الوسطى,104416
المنطقة الغربية,68625
المنطقة الشمالية,34970


In [75]:
# =========================================================
# (4) SAFE TEXT CLEANING + SHOW AS DATAFRAMES
#   - لا نحول: ئ/ؤ/ة/ى (حتى لا نخرب المعنى)
#   - نزيل التشكيل + التطويل (ـ)
#   - نوحّد الألف فقط
#   - نحذف النص إذا كان أرقام فقط
#   - ✅ نحافظ على Region_Name / Region_Folder / City_Folder ... إن وجدت
# =========================================================
print("\n[STEP 4] Safe Cleaning + DataFrame comparison")

TEXT_COL = "Text_TR"

AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
AR_TATWEEL_RE    = re.compile(r"\u0640")

def normalize_arabic_safe(text: str) -> str:
    text = AR_DIACRITICS_RE.sub("", text)   # remove diacritics
    text = AR_TATWEEL_RE.sub("", text)      # remove tatweel (ـ)
    text = re.sub(r"[إأآا]", "ا", text)      # unify alef only
    return text

def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    if not s:
        return ""

    # links / emails / mentions / hashtags
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"\S+@\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#\w+", " ", s)

    # emojis
    s = re.sub(r"[\U00010000-\U0010ffff]", " ", s)

    # keep Arabic/English/digits/spaces
    s = re.sub(r"[^0-9a-z\u0600-\u06FF\s]", " ", s)

    # safe normalize
    s = normalize_arabic_safe(s)

    # reduce repeated letters (جمييييل -> جمييل)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    # حذف إذا كان أرقام فقط
    if re.fullmatch(r"\d+", s):
        return ""

    return s


# ✅ أعمدة تعريفية نريد الحفاظ عليها إن وجدت (عشان تقارير المناطق تشتغل)
META_COLS = ["Region_Name", "Region_Folder", "City_Folder", "Source_File", "__path__"]
meta_existing = [c for c in META_COLS if c in df3.columns]

# BEFORE df (مع الأعمدة التعريفية إن وجدت)
df_before_clean = df3[[TEXT_COL, "Stars_num", "y_bin"] + meta_existing].copy()
df_before_clean = df_before_clean.rename(columns={TEXT_COL: "text_before"})
display(df_before_clean.head(10))

# apply clean
df4 = df3.copy()
df4["text_clean"] = df4[TEXT_COL].apply(clean_text)

# compare df (✅ حافظ على الأعمدة التعريفية)
df_compare = df4[[TEXT_COL, "text_clean", "Stars_num", "y_bin"] + meta_existing].copy()
df_compare = df_compare.rename(columns={TEXT_COL: "text_before"})
display(df_compare.head(20))

# removed empty
df_removed = df_compare[df_compare["text_clean"].str.len() == 0].copy()
print("Removed rows (empty after clean):", len(df_removed))
display(df_removed.head(20))

# final cleaned (no empty) (✅ Region_Name يبقى موجود)
df4 = df_compare[df_compare["text_clean"].str.len() > 0].copy()
print("Final cleaned shape:", df4.shape)
print("Columns:", df4.columns.tolist())
display(df4.head(20))

print("\n[CHECK] safe normalization examples:")
for t in ["سيئ", "سيئة", "سيء", "المكان سيئ جدا", "المكان رائع جدا", "12345", "مطعم 123"]:
    print(f"{t}  ->  {clean_text(t)}")


[STEP 4] Safe Cleaning + DataFrame comparison


,text_before,Stars_num,y_bin,Region_Name,Region_Folder,City_Folder,Source_File,__path__
0,جميلة جدا,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
2,مكان جميل وشعبي,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
16,المكان جميل,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
17,اجواء رائعه جدا,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
19,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
20,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
25,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
27,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
29,مجهود يشكر,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...


,text_before,text_clean,Stars_num,y_bin,Region_Name,Region_Folder,City_Folder,Source_File,__path__
0,جميلة جدا,جميلة جدا,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
2,مكان جميل وشعبي,مكان جميل وشعبي,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
16,المكان جميل,المكان جميل,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
17,اجواء رائعه جدا,اجواء رائعه جدا,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
19,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
20,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
25,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
27,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
29,مجهود يشكر,مجهود يشكر,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...


Removed rows (empty after clean): 2474


,text_before,text_clean,Stars_num,y_bin,Region_Name,Region_Folder,City_Folder,Source_File,__path__
24130,٩,,4.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزة غابة رغدان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
30697,٠١,,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزة غابة رغدان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
39763,100,,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزة غابة رغدان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
65154,١,,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزه غابة خيرة_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
71100,100,,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الأمير محمد بن سعود_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
80873,٢,,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,منتزه الأمير حسام_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
89477,707,,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,مزرعة بساتين اللوز - بني ظبيان_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
100242,٤,,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة جازان,الكورنيش الجنوبي ( الحزام الجنوبي )_textready....,/kaggle/input/datasets/aymanalzahrani7/graduat...
130946,٩,,2.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة جازان,الكورنيش الشمالي_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
131864,١٠٠,,4.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة جازان,شاطئ الشقيق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...


Final cleaned shape: (453702, 9)
Columns: ['text_before', 'text_clean', 'Stars_num', 'y_bin', 'Region_Name', 'Region_Folder', 'City_Folder', 'Source_File', '__path__']


,text_before,text_clean,Stars_num,y_bin,Region_Name,Region_Folder,City_Folder,Source_File,__path__
0,جميلة جدا,جميلة جدا,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
2,مكان جميل وشعبي,مكان جميل وشعبي,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
3,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,رساله اوجهها للي باع لي الطير في الحديقه حسبي ...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
16,المكان جميل,المكان جميل,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
17,اجواء رائعه جدا,اجواء رائعه جدا,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
19,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,منتزة جميل فية العاب اطفال برسوم 55 ريال والوق...,2.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
20,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,جميل بس دقيت مشوار الين وصلته بالنهايه يطلع زي...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
25,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,حديقة جميله جدا ومتوفر فيها محلات تسوق والعاب ...,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
27,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,مبالغين في اسعار الالعاب في المدن الاخري ٥ ريا...,1.0,0,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...
29,مجهود يشكر,مجهود يشكر,5.0,1,المنطقة الجنوبية,( المنطقة الجنوبية ) Google Maps Data - After...,منطقة الباحة,حديقة الفراشة بالمندق_textready.xlsx,/kaggle/input/datasets/aymanalzahrani7/graduat...



[CHECK] safe normalization examples:
سيئ  ->  سيئ
سيئة  ->  سيئة
سيء  ->  سيء
المكان سيئ جدا  ->  المكان سيئ جدا
المكان رائع جدا  ->  المكان رائع جدا
12345  ->  
مطعم 123  ->  مطعم 123


In [76]:
# =========================================================
# (5) SANITY CHECK (Lexicon contradictions) + PER REGION REPORT
# =========================================================
print("\n[STEP 5] Sanity check for star-label consistency (lexicon heuristic)")

# تأكد أن df4 يحتوي الأعمدة المطلوبة
need_cols = ["text_clean", "y_bin", "Stars_num"]
missing = [c for c in need_cols if c not in df4.columns]
if missing:
    raise ValueError(f"Missing required columns in df4: {missing}")

# =========================================================
# Lexicons
# =========================================================
pos_words = set([
    "ممتاز","ممتازه","ممتازة","رائع","رائعة","روعة","جميل","جميلة","جيد","جيدة","جيده",
    "مميز","مميزة","افضل","الأفضل","رهيب","خرافي","فخم","راقي","احترافي","احترافية",
    "ابداع","ابداعي","اسطوري","مذهل","تحفة","خيالي","يجنن","حلو","حلوه","حلوة",
    "شكرا","شكراً","يعطيكم","يعطيك","يعطيكم العافيه","يعطيكم العافية","بيض الله وجهكم",
    "تسلم","تسلمون","مشكور","مشكورين","ممتنين","سعيد","سعيده","فرحت","انبساط","انبسطت",
    "متعاون","متعاونين","تعاون","احترام","محترم","لبق","مهذب","مؤدب","ودود","بشوش",
    "سريع","سريعه","سريعين","سرعة","بدون انتظار","بدون انتضار","فوري","انجاز سريع",
    "نظيف","نظيفه","نظيفة","نظافه","نظافة","مرتب","مرتبين","منظم","نظيف جدا","نظيف جدًا",
    "لذيذ","لذيذه","لذيذة","لذيذ جدا","لذيذ جدًا","شهي","يشهي","طازج","فريش","مقرمش","مضبوط",
    "سعر مناسب","اسعار مناسبة","اسعار مناسبه","رخيص","رخيصة","اقتصادي","مناسب","قيمة ممتازة","يستاهل","يستحق",
    "انصح","انصح فيه","انصح به","انصحكم","انصح الجميع","تجربة رائعة","تجربه رائعه","زيارة موفقة","سأعود","ساعود","برجع",
    "مريح","هادئ","جميل جدا","جميل جدًا","اطلالة رائعة","اطلاله رائعه","ديكور جميل","جلسات جميلة",
    "كفو","تمام","تمام التمام","ولا غلطة","ولا غلطه","على مستوى","فنان","فنااان","مره حلو","مرة حلو","مره ممتاز","مرة ممتاز"
])

neg_words = set([
    "سيء","سيئ","سيئة","سيئه","رديء","رديئة","اسوء","الاسوء","كارثي","فاشل","تعبان","ضعيف","مخيب","مخيبة",
    "لاانصح","لا انصح","ماانصح","ما انصح","لا اوصي","ما اوصي","لن اعود","لن أعود","ما ارجع","مارجع","مستحيل ارجع",
    "تعامل سيء","اسلوب سيء","وقاحة","وقح","قلة ادب","قله ادب","عدم احترام","اهمال","تجاهل","يسحبون عليك",
    "بطئ","بطيء","بطيئ","تاخير","تأخير","متاخر","متأخر","انتظار طويل","زحمة","زحمه","طابور طويل",
    "قذر","وسخ","وصخ","غير نظيف","عدم نظافة","عدم نظافه","وساخة","وساخه","قرف","مقرف",
    "غير لذيذ","سيء الطعم","طعمه سيء","طعم سيء","بارد","محروق","يابس","ماله طعم","مو لذيذ","مو حلو",
    "غالي","غالي جدا","غالي جدًا","مبالغ فيه","سعر مبالغ","اسعار مبالغ فيها","استغلال","نصب","يستغلون",
    "خطأ","غلط","مشكلة","مشاكل","طلبات غلط","طلب غلط","ناقص","ملخبط","فوضى","سيستم خربان",
    "ازعاج","ازعاج قوي","صوت عالي","غير مريح","ضيقة","ضيق","مكان ضيق",
    "زفت","خايس","خايس مره","مره سيء","مرة سيء","ما يستاهل","مايسوى","ما يسوى","زباله"
])

# مطابقة بالاحتواء داخل النص
def lexicon_score(text: str):
    pos = sum(1 for w in pos_words if w in text)
    neg = sum(1 for w in neg_words if w in text)
    return pos - neg, pos, neg

# =========================================================
# Apply scoring (خفيف نسبياً)
# =========================================================
scores = df4["text_clean"].astype(str).apply(lexicon_score)
df4["lex_score"] = scores.apply(lambda x: x[0])
df4["pos_hits"]  = scores.apply(lambda x: x[1])
df4["neg_hits"]  = scores.apply(lambda x: x[2])

# =========================================================
# Contradictions
#   neg label لكن نص إيجابي قوي، pos label لكن نص سلبي قوي
# =========================================================
contrad_neg = df4[(df4["y_bin"] == 0) & (df4["lex_score"] >= 2)].copy()
contrad_pos = df4[(df4["y_bin"] == 1) & (df4["lex_score"] <= -2)].copy()

print("Potential contradictions (neg-stars but positive text):", len(contrad_neg))
print("Potential contradictions (pos-stars but negative text):", len(contrad_pos))

# =========================================================
# Overall samples (خفف العرض)
# =========================================================
show_cols = ["text_before", "text_clean", "Stars_num", "y_bin", "lex_score", "pos_hits", "neg_hits"]
# إذا Region_Name موجود، اعرضه
if "Region_Name" in df4.columns:
    show_cols = ["Region_Name"] + show_cols
if "City_Folder" in df4.columns:
    show_cols = ["City_Folder"] + show_cols

print("\n--- Sample contradictions: neg-stars but positive text ---")
display(contrad_neg[show_cols].head(20))

print("\n--- Sample contradictions: pos-stars but negative text ---")
display(contrad_pos[show_cols].head(20))

# =========================================================
# ✅ PER-REGION REPORT (Counts + Rates)  (بدون تعليق)
# =========================================================
if "Region_Name" in df4.columns:
    print("\n[STEP 5] Per-region contradiction report")

    # إجمالي صفوف كل منطقة
    region_total = df4.groupby("Region_Name").size().rename("total_rows")

    # تناقضات كل منطقة
    region_contra_neg = contrad_neg.groupby("Region_Name").size().rename("contrad_neg")
    region_contra_pos = contrad_pos.groupby("Region_Name").size().rename("contrad_pos")

    df_region_contra = pd.concat([region_total, region_contra_neg, region_contra_pos], axis=1).fillna(0)
    df_region_contra[["contrad_neg","contrad_pos"]] = df_region_contra[["contrad_neg","contrad_pos"]].astype(int)
    df_region_contra["contra_total"] = df_region_contra["contrad_neg"] + df_region_contra["contrad_pos"]

    # نسب
    df_region_contra["contra_rate_%"] = (df_region_contra["contra_total"] / df_region_contra["total_rows"] * 100).round(3)

    df_region_contra = df_region_contra.sort_values("contra_total", ascending=False).reset_index()

    display(df_region_contra)

    # عرض عينات قليلة لكل منطقة (لتفادي التعليق)
    MAX_SAMPLES_PER_REGION = 5
    print(f"\nSamples per region (max {MAX_SAMPLES_PER_REGION} لكل نوع):")

    for region in df_region_contra["Region_Name"].head(10):  # اعرض أعلى 10 مناطق فقط
        a = contrad_neg[contrad_neg["Region_Name"] == region]
        b = contrad_pos[contrad_pos["Region_Name"] == region]

        if len(a) == 0 and len(b) == 0:
            continue

        print(f"\n--- Region: {region} ---")
        if len(a):
            print("neg-stars but positive text:")
            display(a[show_cols].head(MAX_SAMPLES_PER_REGION))
        if len(b):
            print("pos-stars but negative text:")
            display(b[show_cols].head(MAX_SAMPLES_PER_REGION))

# اختياري: استبعاد المتعارضات لتقليل label-noise
# df4 = df4.drop(index=contrad_neg.index.union(contrad_pos.index)).copy()


[STEP 5] Sanity check for star-label consistency (lexicon heuristic)
Potential contradictions (neg-stars but positive text): 7370
Potential contradictions (pos-stars but negative text): 609

--- Sample contradictions: neg-stars but positive text ---


,City_Folder,Region_Name,text_before,text_clean,Stars_num,y_bin,lex_score,pos_hits,neg_hits
87,منطقة الباحة,المنطقة الجنوبية,الفراشه صارت تسد النفس كلها بياعه ماعاد فيها م...,الفراشه صارت تسد النفس كلها بياعه ماعاد فيها م...,2.0,0,2,2,0
126,منطقة الباحة,المنطقة الجنوبية,كئيبه بس حلوه للاطفال,كئيبه بس حلوه للاطفال,2.0,0,2,2,0
132,منطقة الباحة,المنطقة الجنوبية,الفراشة كانت افضل حديقة في المندق لكن ذلحين ما...,الفراشة كانت افضل حديقة في المندق لكن ذلحين ما...,2.0,0,2,2,0
142,منطقة الباحة,المنطقة الجنوبية,تجربتي في المقهي مع الاسف لم يقدم خدمة او جودة...,تجربتي في المقهي مع الاسف لم يقدم خدمة او جودة...,1.0,0,2,4,2
183,منطقة الباحة,المنطقة الجنوبية,مكان مريح وجميل جدا وخاصة للعوائل اللي معهم اط...,مكان مريح وجميل جدا وخاصة للعوائل اللي معهم اط...,1.0,0,3,3,0
284,منطقة الباحة,المنطقة الجنوبية,حطيت نجمه عشان التعليق بس ، انا بتكلم عن الصال...,حطيت نجمه عشان التعليق بس ، انا بتكلم عن الصال...,1.0,0,3,4,1
402,منطقة الباحة,المنطقة الجنوبية,الحديقة جيدة جدا للعائلات الدخول مجانا ، يتوفر...,الحديقة جيدة جدا للعائلات الدخول مجانا ، يتوفر...,1.0,0,4,5,1
441,منطقة الباحة,المنطقة الجنوبية,كانت ممتازة للعائلة ، اما الان,كانت ممتازة للعائلة ، اما الان,1.0,0,2,2,0
467,منطقة الباحة,المنطقة الجنوبية,الحديقة ممتازة للاطفال والعوائل وياليت الشباب ...,الحديقة ممتازة للاطفال والعوائل وياليت الشباب ...,1.0,0,2,2,0
556,منطقة الباحة,المنطقة الجنوبية,مره حلو المكان بس العيب الوحيد ان في جراد واجد...,مره حلو المكان بس العيب الوحيد ان في جراد واجد...,2.0,0,2,2,0



--- Sample contradictions: pos-stars but negative text ---


,City_Folder,Region_Name,text_before,text_clean,Stars_num,y_bin,lex_score,pos_hits,neg_hits
297,منطقة الباحة,المنطقة الجنوبية,حركة سيئه عدم صعود الاطفال الي المسرح الا من ي...,حركة سيئه عدم صعود الاطفال الي المسرح الا من ي...,4.0,1,-2,0,2
408,منطقة الباحة,المنطقة الجنوبية,مشكله الموسيقي عاليه جدا وفرش الملاهي الخارجي ...,مشكله الموسيقي عاليه جدا وفرش الملاهي الخارجي ...,5.0,1,-2,1,3
2535,منطقة الباحة,المنطقة الجنوبية,دورات المياة سيئة جدا,دورات المياة سيئة جدا,5.0,1,-2,0,2
2724,منطقة الباحة,المنطقة الجنوبية,المشكلة في غلي الاسعار المبالغ فيها,المشكلة في غلي الاسعار المبالغ فيها,5.0,1,-2,0,2
2843,منطقة الباحة,المنطقة الجنوبية,السعر مبالغ فيه جدا,السعر مبالغ فيه جدا,4.0,1,-2,0,2
5746,منطقة الباحة,المنطقة الجنوبية,دورات المياه سيئه جدا وايضا عدم توفر اماكن للشراء,دورات المياه سيئه جدا وايضا عدم توفر اماكن للشراء,4.0,1,-2,0,2
8038,منطقة الباحة,المنطقة الجنوبية,المنتزه رائع ولا باس فيه ولكن يوجد فيه مشكلة و...,المنتزه رائع ولا باس فيه ولكن يوجد فيه مشكلة و...,4.0,1,-2,1,3
11843,منطقة الباحة,المنطقة الجنوبية,يعيب المكان دورة المياة جدا سيئه كذلك مواقف ال...,يعيب المكان دورة المياة جدا سيئه كذلك مواقف ال...,4.0,1,-2,0,2
15910,منطقة الباحة,المنطقة الجنوبية,المنتزه حلوا بس مواقف السيارات جدا قليله ودايم...,المنتزه حلوا بس مواقف السيارات جدا قليله ودايم...,5.0,1,-2,1,3
16790,منطقة الباحة,المنطقة الجنوبية,المكان شرح ونظيف صراحة بس الاسعار مبالغ فيها و...,المكان شرح ونظيف صراحة بس الاسعار مبالغ فيها و...,5.0,1,-2,1,3



[STEP 5] Per-region contradiction report


...

,City_Folder,Region_Name,text_before,text_clean,Stars_num,y_bin,lex_score,pos_hits,neg_hits
605830,الشعيبة,المنطقة الغربية,لمن يبحث عن الهدوء والبعد عن الازعاج والزحمة,لمن يبحث عن الهدوء والبعد عن الازعاج والزحمة,5.0,1,-2,0,2
606010,الشعيبة,المنطقة الغربية,الشاطئ لطيف و واسع لكن دورات المياه سيئه,الشاطئ لطيف و واسع لكن دورات المياه سيئه,4.0,1,-2,0,2
606390,الشعيبة,المنطقة الغربية,الحمامات العمومية جدا سيئة,الحمامات العمومية جدا سيئة,4.0,1,-2,0,2
608833,الطايف,المنطقة الغربية,روعه بس الشبكه شوي سيئه ويجب الحذر وقت الامطار,روعه بس الشبكه شوي سيئه ويجب الحذر وقت الامطار,5.0,1,-2,0,2
610988,الطايف,المنطقة الغربية,الزيارة كانت وسط الاسبوع المواقف قليلة جدا سعر...,الزيارة كانت وسط الاسبوع المواقف قليلة جدا سعر...,4.0,1,-2,0,2



--- Region: المنطقة الشمالية ---
neg-stars but positive text:


,City_Folder,Region_Name,text_before,text_clean,Stars_num,y_bin,lex_score,pos_hits,neg_hits
531656,تبوك,المنطقة الشمالية,جميل جدا حطيت نجمه عشان الاغاني,جميل جدا حطيت نجمه عشان الاغاني,1.0,0,2,2,0
531750,تبوك,المنطقة الشمالية,تحتاج افضل مما هي عليه البدايات جميل الان مستو...,تحتاج افضل مما هي عليه البدايات جميل الان مستو...,1.0,0,3,3,0
531783,تبوك,المنطقة الشمالية,لا انصح به ولايوجد اي تعاون من الموظفين والاغا...,لا انصح به ولايوجد اي تعاون من الموظفين والاغا...,1.0,0,2,3,1
531795,تبوك,المنطقة الشمالية,الالعاب للاطفال جميلة واسعارها معقولة بس الاغا...,الالعاب للاطفال جميلة واسعارها معقولة بس الاغا...,1.0,0,2,2,0
531820,تبوك,المنطقة الشمالية,مكان جميل بس ليش كلو مليان مجسمات وارواح حسيت ...,مكان جميل بس ليش كلو مليان مجسمات وارواح حسيت ...,2.0,0,2,3,1


pos-stars but negative text:


,City_Folder,Region_Name,text_before,text_clean,Stars_num,y_bin,lex_score,pos_hits,neg_hits
531832,تبوك,المنطقة الشمالية,الخدمه سيئه للغايه وكل ما جيت علي طاوله قالوا ...,الخدمه سيئه للغايه وكل ما جيت علي طاوله قالوا ...,5.0,1,-2,1,3
534278,تبوك,المنطقة الشمالية,شبكة الاتصال سيئة داخل المول,شبكة الاتصال سيئة داخل المول,4.0,1,-2,0,2
535297,تبوك,المنطقة الشمالية,فقط اشارة الشبكة والهاتف سيئة للغاية,فقط اشارة الشبكة والهاتف سيئة للغاية,5.0,1,-2,0,2
535957,تبوك,المنطقة الشمالية,مكان جيد للتسوق، المشكلة هي ان خدمة الانترنت د...,مكان جيد للتسوق، المشكلة هي ان خدمة الانترنت د...,4.0,1,-2,1,3
536502,تبوك,المنطقة الشمالية,مواقف السيارات سيئة، لكن ما عدا ذلك، كل شيء عل...,مواقف السيارات سيئة، لكن ما عدا ذلك، كل شيء عل...,4.0,1,-2,0,2


In [77]:
# =========================================================
# (6) BALANCE CLASSES (OVERSAMPLING) PER REGION + REPORT
# =========================================================
print("\n[STEP 6] Balancing classes using OVERSAMPLING (per region)")

# لو تبغى تحد حجم البيانات بعد الموازنة لكل منطقة لتجنب التعليق:
# None = بدون حد
MAX_FINAL_PER_REGION = None   # مثال: 30000

if "Region_Name" not in df4.columns:
    raise ValueError("Region_Name غير موجود في df4. تأكد أنك حافظت عليه في Step 4.")

before_stats = (
    df4.groupby(["Region_Name", "y_bin"])
       .size()
       .unstack(fill_value=0)
       .rename(columns={0:"سلبي", 1:"ايجابي"})
       .reset_index()
)
before_stats["total"] = before_stats["سلبي"] + before_stats["ايجابي"]

print("\n[Before balance] per region:")
display(before_stats.sort_values("total", ascending=False))

balanced_parts = []

for region, g in df4.groupby("Region_Name"):
    df_pos = g[g["y_bin"] == 1].copy()
    df_neg = g[g["y_bin"] == 0].copy()

    # إذا منطقة ما فيها أحد الكلاسين، نتجاوزها (أو نخليها كما هي)
    if len(df_pos) == 0 or len(df_neg) == 0:
        # خيار: تجاهل المنطقة لأنها غير قابلة للتوازن
        # أو ضمها بدون توازن:
        # balanced_parts.append(g)
        print(f"  [SKIP] {region}: missing a class (pos={len(df_pos)}, neg={len(df_neg)})")
        continue

    # الهدف: نكبر الأصغر إلى حجم الأكبر (Oversampling)
    target_n = max(len(df_pos), len(df_neg))

    if len(df_pos) < target_n:
        df_pos = resample(df_pos, replace=True, n_samples=target_n, random_state=42)
    if len(df_neg) < target_n:
        df_neg = resample(df_neg, replace=True, n_samples=target_n, random_state=42)

    df_region_bal = pd.concat([df_pos, df_neg], ignore_index=True)\
                      .sample(frac=1, random_state=42)\
                      .reset_index(drop=True)

    # حد أقصى لتجنب تضخم منطقة ضخمة جداً
    if MAX_FINAL_PER_REGION is not None and len(df_region_bal) > MAX_FINAL_PER_REGION:
        df_region_bal = df_region_bal.sample(n=MAX_FINAL_PER_REGION, random_state=42).reset_index(drop=True)

    balanced_parts.append(df_region_bal)

# دمج كل المناطق بعد الموازنة
df_bal = pd.concat(balanced_parts, ignore_index=True)\
          .sample(frac=1, random_state=42)\
          .reset_index(drop=True)

after_stats = (
    df_bal.groupby(["Region_Name", "y_bin"])
          .size()
          .unstack(fill_value=0)
          .rename(columns={0:"سلبي", 1:"ايجابي"})
          .reset_index()
)
after_stats["total"] = after_stats["سلبي"] + after_stats["ايجابي"]

print("\n[After balance] per region:")
display(after_stats.sort_values("total", ascending=False))

print("\n[STEP 6 RESULT] Overall distribution:")
print(df_bal["y_bin"].value_counts().rename({0:"سلبي", 1:"ايجابي"}))
print("[STEP 6 RESULT] df_bal shape:", df_bal.shape)


[STEP 6] Balancing classes using OVERSAMPLING (per region)

[Before balance] per region:


y_bin,Region_Name,سلبي,ايجابي,total
0,المنطقة الجنوبية,19824,114643,134467
1,المنطقة الشرقية,13193,100324,113517
4,المنطقة الوسطى,13538,90807,104345
3,المنطقة الغربية,9316,57294,66610
2,المنطقة الشمالية,4837,29926,34763



[After balance] per region:


y_bin,Region_Name,سلبي,ايجابي,total
0,المنطقة الجنوبية,114643,114643,229286
1,المنطقة الشرقية,100324,100324,200648
4,المنطقة الوسطى,90807,90807,181614
3,المنطقة الغربية,57294,57294,114588
2,المنطقة الشمالية,29926,29926,59852



[STEP 6 RESULT] Overall distribution:
y_bin
سلبي      392994
ايجابي    392994
Name: count, dtype: int64
[STEP 6 RESULT] df_bal shape: (785988, 12)


In [42]:
print("Columns:", df_bal.columns.tolist() if "df_bal" in globals() else df4.columns.tolist())

Columns: ['text_before', 'text_clean', 'Stars_num', 'y_bin']


In [78]:
# =========================================================
# (7) SPLIT TRAIN/VAL/TEST (keep region linked)
# =========================================================
print("\n[STEP 7] Split train/val/test (keep region linked)")

# تأكد من وجود الأعمدة
if "text_clean" not in df_bal.columns or "y_bin" not in df_bal.columns:
    raise ValueError("df_bal لازم يحتوي text_clean و y_bin.")

# اختر عمود المنطقة المناسب
region_col = "Region_Name" if "Region_Name" in df_bal.columns else "Region_Folder"
if region_col not in df_bal.columns:
    raise ValueError("لا يوجد Region_Name ولا Region_Folder داخل df_bal.")

# (اختياري) المدينة
city_col = "City_Folder" if "City_Folder" in df_bal.columns else None

X = df_bal["text_clean"].values
y = df_bal["y_bin"].values.astype(int)
regions = df_bal[region_col].astype(str).values  # <-- ربط المنطقة
cities  = df_bal[city_col].astype(str).values if city_col else None

# Split 80/10/10 مع الحفاظ على ارتباط المنطقة (نمرّرها كـ arrays إضافية)
if cities is not None:
    X_train, X_temp, y_train, y_temp, r_train, r_temp, c_train, c_temp = train_test_split(
        X, y, regions, cities, test_size=0.20, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test, r_val, r_test, c_val, c_test = train_test_split(
        X_temp, y_temp, r_temp, c_temp, test_size=0.50, random_state=42, stratify=y_temp
    )
else:
    X_train, X_temp, y_train, y_temp, r_train, r_temp = train_test_split(
        X, y, regions, test_size=0.20, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test, r_val, r_test = train_test_split(
        X_temp, y_temp, r_temp, test_size=0.50, random_state=42, stratify=y_temp
    )

print("Region column used:", region_col)
print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

# فحص سريع للتأكد أن الربط صحيح
print("\nTest rows per region:")
print(pd.Series(r_test).value_counts().head(10))


[STEP 7] Split train/val/test (keep region linked)
Region column used: Region_Name
Train: 628790 Val: 78599 Test: 78599

Test rows per region:
المنطقة الجنوبية    23065
المنطقة الشرقية     20016
المنطقة الوسطى      18118
المنطقة الغربية     11447
المنطقة الشمالية     5953
Name: count, dtype: int64


In [79]:
# =========================================================
# (8) TOKENIZE + PAD (MAX_LEN=80)
# =========================================================
print("\n[STEP 8] Tokenization + padding")

MAX_LEN = 80
VOCAB_SIZE = 50000
OOV_TOKEN = "<OOV>"

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token=OOV_TOKEN,
    filters="",     # ✅ انت نظفت مسبقاً
    lower=False
)
tokenizer.fit_on_texts(X_train)

def to_padded(texts):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")

X_train_pad = to_padded(X_train)
X_val_pad   = to_padded(X_val)
X_test_pad  = to_padded(X_test)

vocab_used = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print("Vocab used:", vocab_used)
print("X_train_pad:", X_train_pad.shape)


# =========================================================
# (9) BUILD BiLSTM
# =========================================================
print("\n[STEP 9] Build BiLSTM")

EMB_DIM = 128

model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=vocab_used, output_dim=EMB_DIM, mask_zero=True),  # ✅
    layers.SpatialDropout1D(0.2),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="acc"), tf.keras.metrics.AUC(name="auc")]
)

model.summary()


[STEP 8] Tokenization + padding
Vocab used: 50000
X_train_pad: (628790, 80)

[STEP 9] Build BiLSTM


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 80, 128)        │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 80, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,507,137 (24.82 MB)

 Trainable params: 6,507,137 (24.82 MB)

 Non-trainable params: 0 (0.00 B)

In [80]:
# =========================================================
# (10) TRAIN (EPOCHS=18)
# =========================================================
print("\n[STEP 10] Train model")

EPOCHS = 18
BATCH_SIZE = 256

cb = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)



[STEP 10] Train model
Epoch 1/18
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 46s 18ms/step - acc: 0.8786 - auc: 0.9388 - loss: 0.3067 - val_acc: 0.9221 - val_auc: 0.9731 - val_loss: 0.2060
Epoch 2/18
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - acc: 0.9322 - auc: 0.9788 - loss: 0.1812 - val_acc: 0.9351 - val_auc: 0.9807 - val_loss: 0.1740
Epoch 3/18
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - acc: 0.9470 - auc: 0.9870 - loss: 0.1407 - val_acc: 0.9417 - val_auc: 0.9832 - val_loss: 0.1620
Epoch 4/18
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - acc: 0.9544 - auc: 0.9905 - loss: 0.1193 - val_acc: 0.9464 - val_auc: 0.9848 - val_loss: 0.1546
Epoch 5/18
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - acc: 0.9589 - auc: 0.9922 - loss: 0.1071 - val_acc: 0.9460 - val_auc: 0.9846 - val_loss: 0.1611
Epoch 6/18
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - acc: 0.9616 - auc: 0.9932 - loss: 0.1000 - val_acc: 0.9486 - val_auc: 0.9851 - val_loss: 0.1624
Epoch 7/18
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/st

In [81]:
# =========================================================
# (11) REPORTS (TEST) + PER REGION REPORTS
# =========================================================
print("\n[STEP 11] Test evaluation + reports (overall + per region)")

# تنبؤات
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# =========================
# Overall metrics
# =========================
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("\n[OVERALL]")
print("Test Accuracy:", round(acc, 4))
print("Test ROC-AUC :", round(auc, 4))

print("\nClassification Report (overall):")
print(classification_report(y_test, y_pred, target_names=["سلبي", "ايجابي"], digits=4))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix (overall):")
print(cm)

# =========================
# Per-region reports
# =========================
if "r_test" not in globals():
    raise ValueError("r_test غير موجود. تأكد أنك استخدمت Step 7 الذي يحفظ r_test (المناطق).")

df_test_region = pd.DataFrame({
    "Region": pd.Series(r_test).astype(str).values,
    "y_true": y_test,
    "y_pred": y_pred,
    "p_pos": y_prob
})

print("\n[PER REGION] Summary (accuracy/AUC + counts)")

rows = []
for region, g in df_test_region.groupby("Region"):
    n = len(g)
    acc_r = (g["y_true"] == g["y_pred"]).mean()
    auc_r = None
    if g["y_true"].nunique() == 2:
        auc_r = roc_auc_score(g["y_true"], g["p_pos"])
    rows.append([region, n, acc_r, auc_r])

df_region_summary = pd.DataFrame(rows, columns=["Region", "n_test", "accuracy", "auc"])\
                      .sort_values("n_test", ascending=False)\
                      .reset_index(drop=True)

display(df_region_summary)

print("\n[PER REGION] Detailed reports + confusion matrices")

for region, g in df_test_region.groupby("Region"):
    print("\n" + "="*70)
    print(f"Region: {region} | n_test={len(g)}")
    print("="*70)

    print("\nClassification Report:")
    print(classification_report(
        g["y_true"], g["y_pred"],
        target_names=["سلبي", "ايجابي"],
        digits=4,
        zero_division=0
    ))

    cm_r = confusion_matrix(g["y_true"], g["y_pred"])
    print("Confusion Matrix:")
    print(cm_r)


[STEP 11] Test evaluation + reports (overall + per region)

[OVERALL]
Test Accuracy: 0.9554
Test ROC-AUC : 0.9898

Classification Report (overall):
              precision    recall  f1-score   support

        سلبي     0.9534    0.9577    0.9555     39300
      ايجابي     0.9575    0.9532    0.9553     39299

    accuracy                         0.9554     78599
   macro avg     0.9555    0.9554    0.9554     78599
weighted avg     0.9555    0.9554    0.9554     78599


Confusion Matrix (overall):
[[37638  1662]
 [ 1840 37459]]

[PER REGION] Summary (accuracy/AUC + counts)


,Region,n_test,accuracy,auc
0,المنطقة الجنوبية,23065,0.955040,0.988989
1,المنطقة الشرقية,20016,0.956635,0.990380
2,المنطقة الوسطى,18118,0.957225,0.990978
3,المنطقة الغربية,11447,0.947759,0.987292
4,المنطقة الشمالية,5953,0.962372,0.991868



[PER REGION] Detailed reports + confusion matrices

Region: المنطقة الجنوبية | n_test=23065

Classification Report:
              precision    recall  f1-score   support

        سلبي     0.9540    0.9562    0.9551     11535
      ايجابي     0.9561    0.9539    0.9550     11530

    accuracy                         0.9550     23065
   macro avg     0.9550    0.9550    0.9550     23065
weighted avg     0.9550    0.9550    0.9550     23065

Confusion Matrix:
[[11030   505]
 [  532 10998]]

Region: المنطقة الشرقية | n_test=20016

Classification Report:
              precision    recall  f1-score   support

        سلبي     0.9558    0.9582    0.9570     10078
      ايجابي     0.9575    0.9550    0.9563      9938

    accuracy                         0.9566     20016
   macro avg     0.9566    0.9566    0.9566     20016
weighted avg     0.9566    0.9566    0.9566     20016

Confusion Matrix:
[[9657  421]
 [ 447 9491]]

Region: المنطقة الشمالية | n_test=5953

Classification Report:
       

In [26]:
# =========================================================
# (12) ERROR ANALYSIS (Hard mistakes)
# =========================================================
print("\n[STEP 12] Error analysis samples (most confident wrong)")

df_test = pd.DataFrame({
    "text_clean": X_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "p_pos": y_prob
})

wrong = df_test[df_test["y_true"] != df_test["y_pred"]].copy()
wrong["confidence"] = np.abs(wrong["p_pos"] - 0.5)
wrong = wrong.sort_values("confidence", ascending=False)

print("Wrong predictions:", len(wrong))
display(wrong.head(30).reset_index(drop=True))


[STEP 12] Error analysis samples (most confident wrong)
Wrong predictions: 542


,text_clean,y_true,y_pred,p_pos,confidence
0,يوجد بها مرجيحات، دورات المياة يتم الاعتناء به...,0,1,1.000000e+00,0.500000
1,كويس ولكن الله يسترنا بستره ويستر كل عفيفة وعف...,0,1,1.000000e+00,0.500000
2,المكان جميل والطعام حلو بس صراحة الكاشيرة جدا ...,0,1,1.000000e+00,0.500000
3,المكان جميل ش سعر الدخول جدا غالي ب ٣٥ للفرد و...,0,1,1.000000e+00,0.500000
4,بصراحه ولا بحق المشوار من جده للطائف لزيارتهم ...,0,1,1.000000e+00,0.500000
5,دورة المياة بفلوس ليش؟؟ سلامات؟ احسن لي اروح ا...,0,1,1.000000e+00,0.500000
6,يحتاج لاعادة تاهيل كامل لنرتقي للاحداث الرياضي...,0,1,1.000000e+00,0.500000
7,يحتاج لاعادة تاهيل كامل لنرتقي للاحداث الرياضي...,0,1,1.000000e+00,0.500000
8,المكان جميل جدا واختيار اكثر من رائع ولكن المو...,0,1,1.000000e+00,0.500000
9,زرتها الصباح الصراحة مره هادية و نا فيها نمل م...,1,0,3.050445e-08,0.500000


In [29]:
# ========= Inference helpers =========
def prepare_texts_for_model(texts):
    # texts: list[str]
    cleaned = [clean_text(t) for t in texts]
    seq = tokenizer.texts_to_sequences(cleaned)
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    return cleaned, pad

def predict_texts(texts, threshold=0.5):
    cleaned, X_pad = prepare_texts_for_model(texts)
    probs = model.predict(X_pad, verbose=0).ravel()
    preds = (probs >= threshold).astype(int)

    out = pd.DataFrame({
        "text_raw": texts,
        "text_clean": cleaned,
        "p_positive": probs,
        "pred_label": np.where(preds==1, "ايجابي", "سلبي")
    }).sort_values("p_positive", ascending=False).reset_index(drop=True)

    return out

# ========= Try your own examples =========
samples = [
    "المكان ممتاز والخدمة سريعة 😍",
    "سيء جدا وما أنصح فيه أبدا",
    "الاسعار غالية والخدمة بطيئة",
    "جميل ونظيف وتعامل راقي",
    "سيء"
]

df_pred = predict_texts(samples, threshold=0.5)
display(df_pred)

,text_raw,text_clean,p_positive,pred_label
0,المكان ممتاز والخدمة سريعة 😍,المكان ممتاز والخدمة سريعة,1.000000e+00,ايجابي
1,جميل ونظيف وتعامل راقي,جميل ونظيف وتعامل راقي,9.999999e-01,ايجابي
2,الاسعار غالية والخدمة بطيئة,الاسعار غالية والخدمة بطيئة,2.940662e-04,سلبي
3,سيء,سيء,1.064189e-04,سلبي
4,سيء جدا وما أنصح فيه أبدا,سيء جدا وما انصح فيه ابدا,9.851259e-08,سلبي


In [82]:
# =========================================================
# SAVE MODEL
# =========================================================
print("\n[SAVE] Saving model...")

MODEL_PATH = "/kaggle/working/bilstm_sentiment_model.keras"

model.save(MODEL_PATH)

print("Model saved at:", MODEL_PATH)


[SAVE] Saving model...
Model saved at: /kaggle/working/bilstm_sentiment_model.keras


In [83]:
import pickle

TOKENIZER_PATH = "/kaggle/working/tokenizer.pkl"

with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved.")

Tokenizer saved.


In [84]:
import json

config = {
    "MAX_LEN": MAX_LEN,
    "VOCAB_SIZE": VOCAB_SIZE,
    "EMB_DIM": EMB_DIM
}

CONFIG_PATH = "/kaggle/working/model_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Config saved.")

Config saved.
